# Word Embedding: Dipelajari Bersama Tugas vs Pralatih (GloVe)

**Buku:** *Deep Learning dengan Python* — Bab 6, Deep Learning untuk Teks dan Sekuens

Membandingkan embedding yang dipelajari dari nol dengan embedding GloVe pralatih ketika data latih hanya 200 ulasan.

> Diadaptasi dari notebook pendamping *Deep Learning with Python* (F. Chollet, Manning; kode berlisensi MIT) dan diperbarui ke **Keras 3** untuk buku *Deep Learning dengan Python: Konsep, Kode, dan Penerapan Modern* (B. S. Negara).

**Lingkungan:** Google Colab (GPU disarankan), Python 3.10+, Keras 3. Kode bersifat *backend-agnostik*: dapat dijalankan dengan backend TensorFlow, JAX, atau PyTorch.

**Perubahan utama dari versi asli (Keras 2.0.8):**
- `Tokenizer` + `pad_sequences` lama → `TextVectorization(output_sequence_length=...)` dan `keras.utils.pad_sequences`.
- Argumen `input_length` pada `Embedding` (dihapus di Keras 3) → `keras.Input(shape=(panjang,))`.
- Matriks GloVe dimuat lewat `embeddings_initializer=Constant(...)` + `trainable=False`.
- Data mentah IMDB dan GloVe diunduh otomatis (tanpa path lokal).

In [ ]:
import os
# Pilih backend: "tensorflow", "jax", atau "torch"
os.environ["KERAS_BACKEND"] = "tensorflow"

import keras
print("Keras", keras.__version__, "| backend:", keras.backend.backend())

## 1. Embedding yang dipelajari bersama tugas

In [ ]:
from keras import layers
from keras.datasets import imdb

MAKS_KATA, PANJANG = 10000, 20
(x_latih, y_latih), (x_uji, y_uji) = imdb.load_data(num_words=MAKS_KATA)
x_latih = keras.utils.pad_sequences(x_latih, maxlen=PANJANG)
x_uji = keras.utils.pad_sequences(x_uji, maxlen=PANJANG)

model = keras.Sequential([
    keras.Input(shape=(PANJANG,), dtype="int64"),
    layers.Embedding(MAKS_KATA, 8),
    layers.Flatten(),
    layers.Dense(1, activation="sigmoid"),
])
model.compile(optimizer="rmsprop", loss="binary_crossentropy", metrics=["accuracy"])
model.summary()
riwayat = model.fit(x_latih, y_latih, epochs=10, batch_size=32, validation_split=0.2)

Model ini memperlakukan setiap kata secara terpisah (tanpa urutan), sehingga "film ini bom" dan "film ini *the bomb*" sulit dibedakan. Lapisan RNN/konvolusi 1D di notebook berikutnya mengatasinya.

## 2. Dari teks mentah: unduh IMDB asli

In [ ]:
import os, tarfile, pathlib, numpy as np

def siapkan_aclimdb(folder="data"):
    """Siapkan data/aclImdb/{train,test}/{neg,pos,unsup}/*.txt.
    Sumber utama: server Stanford. Bila tidak dapat dihubungi, unduh salinan
    resmi yang sama dari Hugging Face (stanfordnlp/imdb)."""
    akar = pathlib.Path(folder) / "aclImdb"
    if (akar / "train" / "pos").exists():
        return
    try:
        jalur = keras.utils.get_file(
            "aclImdb_v1.tar.gz",
            origin="https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz")
        with tarfile.open(jalur) as berkas_tar:
            berkas_tar.extractall(folder)
    except Exception as galat:
        print("Server Stanford tidak dapat dihubungi:", galat)
        print("Memakai salinan Hugging Face (stanfordnlp/imdb) ...")
        from datasets import load_dataset
        ds = load_dataset("stanfordnlp/imdb")
        for bagian, nama in [("train", "train"), ("test", "test"), ("unsupervised", "train")]:
            for i, contoh in enumerate(ds[bagian]):
                jenis = "unsup" if bagian == "unsupervised" else ("pos" if contoh["label"] == 1 else "neg")
                tujuan = akar / nama / jenis
                tujuan.mkdir(parents=True, exist_ok=True)
                (tujuan / f"{bagian}_{i}.txt").write_text(contoh["text"], encoding="utf-8")

siapkan_aclimdb()

def baca_folder(folder):
    teks, label = [], []
    for jenis, nilai in [("neg", 0), ("pos", 1)]:
        for p in sorted((pathlib.Path(folder) / jenis).glob("*.txt")):
            teks.append(p.read_text(encoding="utf-8"))
            label.append(nilai)
    return teks, np.array(label)

teks_latih, label_latih = baca_folder("data/aclImdb/train")
teks_uji, label_uji = baca_folder("data/aclImdb/test")
print(len(teks_latih), "ulasan latih |", len(teks_uji), "ulasan uji")

## 3. Tokenisasi dan pembagian
Hanya **200** ulasan dipakai untuk melatih, 10.000 untuk validasi.

In [ ]:
PANJANG, MAKS_KATA, N_LATIH, N_VAL = 100, 10000, 200, 10000

rng = np.random.default_rng(42)
urutan = rng.permutation(len(teks_latih))       # data asli terurut (negatif dulu): wajib diacak
teks_latih = [teks_latih[i] for i in urutan]
label_latih = label_latih[urutan]

vektorisasi = layers.TextVectorization(max_tokens=MAKS_KATA, output_sequence_length=PANJANG)
vektorisasi.adapt(teks_latih)                   # hanya teks dari folder latih (tanpa label)
data = keras.ops.convert_to_numpy(vektorisasi(teks_latih))

x_latih, y_latih = data[:N_LATIH], label_latih[:N_LATIH]
x_val, y_val = data[N_LATIH:N_LATIH + N_VAL], label_latih[N_LATIH:N_LATIH + N_VAL]
x_uji = keras.ops.convert_to_numpy(vektorisasi(teks_uji))
print(x_latih.shape, x_val.shape, x_uji.shape)

## 4. Unduh GloVe (±822 MB) dan bangun matriks embedding

In [ ]:
import zipfile

# Sumber utama Stanford; bila tidak dapat dihubungi, pakai salinan resmi di Hugging Face
try:
    jalur_glove = keras.utils.get_file(
        "glove.6B.zip", origin="https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip")
except Exception as galat:
    print("Server Stanford tidak dapat dihubungi:", galat, "-> memakai Hugging Face")
    jalur_glove = keras.utils.get_file(
        "glove.6B.zip", origin="https://huggingface.co/stanfordnlp/glove/resolve/main/glove.6B.zip")
indeks_embedding = {}
with zipfile.ZipFile(jalur_glove) as zf:
    with zf.open("glove.6B.100d.txt") as f:
        for baris in f:
            nilai = baris.decode("utf-8").split()
            indeks_embedding[nilai[0]] = np.asarray(nilai[1:], dtype="float32")
print(len(indeks_embedding), "vektor kata")

DIM = 100
kosakata = vektorisasi.get_vocabulary()          # indeks 0 = padding, 1 = [UNK]
matriks = np.zeros((MAKS_KATA, DIM))
for i, kata in enumerate(kosakata):
    vektor = indeks_embedding.get(kata)
    if vektor is not None:
        matriks[i] = vektor                      # kata yang tidak ditemukan tetap nol

In [ ]:
import matplotlib.pyplot as plt

def plot_riwayat(riwayat, metrik="accuracy", judul=""):
    h = riwayat.history
    epoch = range(1, len(h["loss"]) + 1)
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].plot(epoch, h[metrik], "o", ms=3, label="Latih")
    ax[0].plot(epoch, h["val_" + metrik], "-", label="Validasi")
    ax[0].set(xlabel="Epoch", ylabel=metrik, title=judul)
    ax[1].plot(epoch, h["loss"], "o", ms=3, label="Latih")
    ax[1].plot(epoch, h["val_loss"], "-", label="Validasi")
    ax[1].set(xlabel="Epoch", ylabel="Loss", title=judul)
    for a in ax: a.legend()
    plt.tight_layout(); plt.show()

def bangun(embedding_pralatih=None):
    if embedding_pralatih is not None:
        emb = layers.Embedding(MAKS_KATA, DIM, trainable=False,
                               embeddings_initializer=keras.initializers.Constant(embedding_pralatih))
    else:
        emb = layers.Embedding(MAKS_KATA, DIM)
    model = keras.Sequential([keras.Input(shape=(PANJANG,), dtype="int64"), emb,
                              layers.Flatten(),
                              layers.Dense(32, activation="relu"),
                              layers.Dense(1, activation="sigmoid")])
    model.compile(optimizer="rmsprop", loss="binary_crossentropy", metrics=["accuracy"])
    return model

model_glove = bangun(matriks)
r1 = model_glove.fit(x_latih, y_latih, epochs=10, batch_size=32, validation_data=(x_val, y_val))
plot_riwayat(r1, judul="GloVe pralatih (dibekukan)")

model_nol = bangun()
r2 = model_nol.fit(x_latih, y_latih, epochs=10, batch_size=32, validation_data=(x_val, y_val))
plot_riwayat(r2, judul="Embedding dari nol")

In [ ]:
print(f"Akurasi uji GloVe   : {model_glove.evaluate(x_uji, label_uji, verbose=0)[1]:.2%}")
print(f"Akurasi uji dari nol: {model_nol.evaluate(x_uji, label_uji, verbose=0)[1]:.2%}")

Dengan hanya 200 contoh, hasilnya bervariasi tergantung sampel yang terpilih. Coba ubah `N_LATIH` menjadi 2.000 atau 20.000 dan amati kapan embedding dari nol menyusul.